In [ ]:
# Installer timm
!pip install timm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.1 MB/s eta 0:00:00


In [ ]:
# Installer timm si necessaire
#!pip install timm

import os
import torch
import numpy as np
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score
import timm
from torch.nn.init import xavier_uniform_
from PIL import Image
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt

# Constants
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_CLASSES = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 10
PREPROCESSED_DIR = "/content/drive/MyDrive/preprocessed_eurosat"
dataset_fraction = 0.1

# Dataset class (Corrected to accept images directly, not image paths)
class EuroSATDataset(Dataset):
    def __init__(self, images, labels, transform=None): # Removed image_paths and using images directly
        self.images = images # Using images directly
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx]).convert('RGB') # Using Image.fromarray
        if self.transform:
            img = self.transform(img)
        label = self.labels[idx]
        return img, label

# Load or preprocess the dataset
def load_or_preprocess_data(dataset_fraction=1.0, train_split=0.7, val_split=0.15, test_split=0.15):
    if train_split + val_split + test_split != 1.0:
        raise ValueError("Train, validation, and test splits must sum to 1.0")

    if os.path.exists(PREPROCESSED_DIR):
        print("Loading preprocessed data from directory...")
        train_data = np.load(os.path.join(PREPROCESSED_DIR, 'train_data.npz'))
        val_data = np.load(os.path.join(PREPROCESSED_DIR, 'val_data.npz'))
        test_data = np.load(os.path.join(PREPROCESSED_DIR, 'test_data.npz'))

        train_images, train_labels = train_data['images'], train_data['labels']
        val_images, val_labels = val_data['images'], val_data['labels']
        test_images, test_labels = test_data['images'], test_data['labels']

    else:
        print("Preprocessing data...")
        dataset = tfds.load("eurosat/rgb", split=['train'], as_supervised=True)
        all_images, all_labels = [], []

        for img, label in tfds.as_numpy(dataset[0]):
            all_images.append(img)
            all_labels.append(label)

        all_images = np.array(all_images)
        all_labels = np.array(all_labels)

        # Shuffle the dataset
        indices = np.arange(len(all_images))
        np.random.shuffle(indices)
        all_images = all_images[indices]
        all_labels = all_labels[indices]

        # Reduce dataset size for quick experiments
        total_samples = int(len(all_images) * dataset_fraction)
        all_images = all_images[:total_samples]
        all_labels = all_labels[:total_samples]

        # Split dataset
        train_end = int(len(all_images) * train_split)
        val_end = train_end + int(len(all_images) * val_split)

        train_images, train_labels = all_images[:train_end], all_labels[:train_end]
        val_images, val_labels = all_images[train_end:val_end], all_labels[train_end:val_end]
        test_images, test_labels = all_images[val_end:], all_labels[val_end:]

        # Save preprocessed data
        os.makedirs(PREPROCESSED_DIR, exist_ok=True)
        np.savez(os.path.join(PREPROCESSED_DIR, 'train_data.npz'), images=train_images, labels=train_labels)
        np.savez(os.path.join(PREPROCESSED_DIR, 'val_data.npz'), images=val_images, labels=val_labels)
        np.savez(os.path.join(PREPROCESSED_DIR, 'test_data.npz'), images=test_images, labels=test_labels)

    return train_images, train_labels, val_images, val_labels, test_images, test_labels


# Transformation des données avec augmentation
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation Data Transforms
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Dataset personnalisé (Corrected - Removed redundant definition)
# class EuroSATDataset(Dataset): # Removed redundant definition here

# ----------------------------------------
# Modèle avec Late-Level Fusion
# ----------------------------------------

class LateFusionWithAttention(nn.Module):
    def __init__(self, num_classes):
        super(LateFusionWithAttention, self).__init__()
        # Charger les modèles Swin et EfficientNet
        self.swin = timm.create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=num_classes)
        self.efficientnet = timm.create_model('efficientnet_b4', pretrained=True, num_classes=num_classes)

        # Mécanisme d'attention pour pondérer dynamiquement les prédictions
        self.attention = nn.Sequential(
            nn.Linear(num_classes * 2, num_classes),
            nn.ReLU(),
            nn.Linear(num_classes, 2),  # Pondérations pour les deux branches
            nn.Softmax(dim=1)
        )

        # Initialisation des poids
        for module in self.attention.modules():
            if isinstance(module, nn.Linear):
                xavier_uniform_(module.weight)

    def forward(self, x):
        swin_out = self.swin(x)  # Sortie Swin
        eff_out = self.efficientnet(x)  # Sortie EfficientNet

        # Concaténer les prédictions
        combined = torch.cat((swin_out, eff_out), dim=1)

        # Appliquer l'attention
        weights = self.attention(combined)
        swin_weight, eff_weight = weights[:, 0].unsqueeze(1), weights[:, 1].unsqueeze(1)

        # Pondération dynamique des prédictions
        fused_out = swin_weight * swin_out + eff_weight * eff_out
        return fused_out

In [ ]:
# Part 2: Training and Testing

# Training, Validation, and Test Functions
from tqdm import tqdm  # Assurez-vous que tqdm est importé

best_val_acc = 0.0
patience_counter = 0
early_stopping_patience = 5

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in tqdm(dataloader, desc="Training"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Validation"):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def test_model(model, dataloader, device):
    model.eval()
    predictions = []
    true_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Testing"):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    return predictions, true_labels

# Main Training Loop

def train_model(model, train_dataloader, val_dataloader, criterion, optimizer, scheduler, device, epochs, early_stopping_patience):
    global best_val_acc, patience_counter

    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")

        # Train phase
        train_loss, train_acc = train_one_epoch(model, train_dataloader, criterion, optimizer, device)
        print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.4f}")

        # Validation phase
        val_loss, val_acc = validate_one_epoch(model, val_dataloader, criterion, device)
        print(f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.4f}")

        # Learning rate scheduler step
        scheduler.step(val_loss)

        # Early stopping check
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), "best_late_model.pth")  # Save the best model
            print("Best late fusion model saved!")
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print("Early stopping triggered.")
                break

# Instantiate and run training
if __name__ == "__main__":
    # Assuming train_dataloader, val_dataloader, and test_dataloader are already defined
    train_images, train_labels, val_images, val_labels, test_images, test_labels = load_or_preprocess_data(dataset_fraction)

    train_dataset = EuroSATDataset(train_images, train_labels, train_transforms)
    val_dataset = EuroSATDataset(val_images, val_labels, val_transforms)
    test_dataset = EuroSATDataset(test_images, test_labels, val_transforms)

    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

    model = LateFusionWithAttention(NUM_CLASSES).to(DEVICE)

    # Phase 1: Freeze backbone layers
    for param in model.swin.parameters():
        param.requires_grad = False
    for param in model.efficientnet.parameters():
        param.requires_grad = False

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, verbose=True)

    print("Starting Phase 1: Training classifier head...")
    train_model(model, train_dataloader, val_dataloader, criterion, optimizer, scheduler, DEVICE, epochs=EPOCHS, early_stopping_patience=5)

    # Phase 2: Unfreeze backbone layers
    for param in model.swin.parameters():
        param.requires_grad = True
    for param in model.efficientnet.parameters():
        param.requires_grad = True

    optimizer = optim.Adam(model.parameters(), lr=1e-4)  # Lower learning rate for fine-tuning

    print("Starting Phase 2: Fine-tuning entire model...")
    train_model(model, train_dataloader, val_dataloader, criterion, optimizer, scheduler, DEVICE, epochs=EPOCHS, early_stopping_patience=5)

    # Test phase
    print("Testing the best model...")
    model.load_state_dict(torch.load("best_late_model.pth"))
    predictions, true_labels = test_model(model, test_dataloader, DEVICE)

    print("Classification Report:")
    print(classification_report(true_labels, predictions))


Preprocessing data...


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

...

Train Loss: 0.0201, Train Accuracy: 0.9952


Validation: 100%|██████████| 7/7 [00:13<00:00,  1.86s/it]


Val Loss: 0.2030, Val Accuracy: 0.9506
Early stopping triggered.
Testing the best model...


Testing: 100%|██████████| 7/7 [00:13<00:00,  1.88s/it]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.94      0.95        51
           1       1.00      0.98      0.99        44
           2       0.85      0.92      0.89        51
           3       0.97      1.00      0.99        38
           4       0.96      1.00      0.98        45
           5       0.95      0.88      0.91        24
           6       0.88      0.90      0.89        49
           7       1.00      0.96      0.98        49
           8       1.00      0.96      0.98        24
           9       1.00      0.97      0.98        31

    accuracy                           0.95       406
   macro avg       0.96      0.95      0.95       406
weighted avg       0.95      0.95      0.95       406



In [ ]:
pip install tensorflow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.9/644.9 MB 595.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.31.1
    Uninstalling protobuf-6.31.1:
      Successfully uninstalled protobuf-6.31.1
